<a href="https://colab.research.google.com/github/AR-Ashik-9997/Amazon-Reviews-for-Sentiment-Analysis/blob/main/Amazon_Reviews_for_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 76.0 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [ ]:
import re
import torch
import pickle
import nltk
import pandas as pd
import numpy as np
import torch.nn as nn
import torch.optim as optim
from nltk.corpus import stopwords
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from torch.utils.data import Dataset, DataLoader
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
labels=[]
texts=[]

with open("/content/train.ft.txt",'r',encoding='utf-8') as file:
  for line in file:
    parts=line.strip().split(' ',1)
    if len(parts)==2:
      label=parts[0].replace('__label__','')
      text=parts[1]

      labels.append(int(label))
      texts.append(text)
df=pd.DataFrame({
    'label':labels,
    'text':texts
})

In [ ]:
df['sentiment']=df['label'].map({1:'negative',2:'positive'})

## Text Preprocessing & Vocabulary

In [ ]:
df['text'].head()

,text
0,Stuning even for the non-gamer: This sound tra...
1,The best soundtrack ever to anything.: I'm rea...
2,Amazing!: This soundtrack is my favorite music...
3,Excellent Soundtrack: I truly like this soundt...
4,"Remember, Pull Your Jaw Off The Floor After He..."


In [ ]:
def clean_text(text,remove_stopwords=False):
  text=text.lower()
  text=re.sub(r'[^a-z\s]','',text)
  text=re.sub(r'\s+',' ', text).strip()
  if remove_stopwords:
    stop_words=set(stopwords.words('english'))
    words=text.split()
    text=' '.join([word for word in words if word not in stop_words])
  return text

In [ ]:
df['clean_text']=df['text'].apply(lambda x: clean_text(x,remove_stopwords=True))

In [ ]:
vocab={"<PAD>":0,"<UNK>":1}
all_words=[]
for text in df['clean_text']:
  all_words.extend(text.split())
word_counts=Counter(all_words)
for word,_ in word_counts.items():
  vocab[word]=len(vocab)
print(f"Unique vocabulary size:{len(vocab)}")

Unique vocabulary size:77767


## Numerical Sequence and Padding

In [ ]:
def text_to_sequence(text,vocab):
  return [vocab.get(word,vocab['<UNK>']) for word in text.split() ]
df['sequence']=df['clean_text'].apply(lambda z: text_to_sequence(z,vocab))

max_len=12

def pad_sequence(seq,max_len):
  if len(seq)<max_len:
    return seq+[vocab['<PAD>']]*(max_len-len(seq))
  else:
    return seq[:max_len]
df['padding']=df['sequence'].apply(lambda s: pad_sequence(s,max_len))


## Create PyTorch Dataset ও DataLoader

In [ ]:
class SentimentDataset(Dataset):
  def __init__(self,sequence,labels):
    self.sequence=torch.tensor(sequence,dtype=torch.long)
    self.labels=torch.tensor(labels,dtype=torch.long)-1
  def __len__(self):
    return len(self.labels)
  def __getitem__(self, idx):
    return self.sequence[idx],self.labels[idx]


## Train test split

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(list(df['padding']),list(df['label']),test_size=0.2,random_state=42)

train_dataset=SentimentDataset(X_train,y_train)
test_dataset=SentimentDataset(X_test,y_test)

train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False)

## Bi-LSTM Model Architechture

In [ ]:
class BiLSTMClassification(nn.Module):
  def __init__(self, vocab_size, embedding_dim, hidden_size, output_dim, pad_idx):
    super().__init__()

    self.embedding=nn.Embedding(vocab_size,embedding_dim,padding_idx=pad_idx)
    self.lstm=nn.LSTM(embedding_dim,hidden_size,batch_first=True,bidirectional=True)
    self.fc=nn.Linear(hidden_size*2,output_dim)
    self.dropout=nn.Dropout(0.5)

  def forward(self,text):
    embedded=self.dropout(self.embedding(text))
    lstm_out,(hidden,cell)=self.lstm(embedded)

    hidden_cat=torch.cat((hidden[-2,:,:],hidden[-1,:,:]),dim=1)
    dense_out=self.dropout(hidden_cat)
    output=self.fc(dense_out)
    return output

## Model Initialization

In [ ]:
EPOCHS=15
VOCAB_SIZE=len(vocab)
EMBEDDING_DIM = 64
HIDDEN_SIZE = 32
OUTPUT_DIM = 2 # Positive / Negative
PAD_IDX = vocab["<PAD>"]
model=BiLSTMClassification(VOCAB_SIZE,EMBEDDING_DIM,HIDDEN_SIZE,OUTPUT_DIM,PAD_IDX)
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)

## Model Training

In [ ]:
model = model.to(device)

for epoch in range(EPOCHS):
  model.train()
  epoch_loss = 0
  correct_train = 0
  total_train = 0

  for batch_x, batch_y in train_loader:
    batch_x, batch_y = batch_x.to(device), batch_y.to(device)
    optimizer.zero_grad()

    outputs = model(batch_x)
    loss = criterion(outputs, batch_y)
    loss.backward()

    optimizer.step()

    epoch_loss += loss.item()
    _, predicted = torch.max(outputs, 1)
    total_train += batch_y.size(0)
    correct_train += (predicted == batch_y).sum().item()

  train_loss = epoch_loss / len(train_loader)
  train_acc = 100 * correct_train / total_train


  model.eval()
  val_loss = 0
  correct_val = 0
  total_val = 0

  with torch.no_grad():
    for batch_x, batch_y in test_loader:
      batch_x, batch_y = batch_x.to(device), batch_y.to(device)
      outputs = model(batch_x)
      loss = criterion(outputs, batch_y)

      val_loss += loss.item()

      _, predicted = torch.max(outputs, 1)
      total_val += batch_y.size(0)
      correct_val += (predicted == batch_y).sum().item()

  val_loss = val_loss / len(test_loader)
  val_acc = 100 * correct_val / total_val

  print(f"Epoch [{epoch+1}/{EPOCHS}] | "
          f"Train Loss: {train_loss:.4f} - Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.2f}%")

Epoch [1/15] | Train Loss: 0.6461 - Train Acc: 61.11% | Val Loss: 0.5494 - Val Acc: 72.37%
Epoch [2/15] | Train Loss: 0.5597 - Train Acc: 71.04% | Val Loss: 0.4886 - Val Acc: 75.81%
Epoch [3/15] | Train Loss: 0.5048 - Train Acc: 75.16% | Val Loss: 0.4920 - Val Acc: 77.35%
Epoch [4/15] | Train Loss: 0.4674 - Train Acc: 77.67% | Val Loss: 0.4390 - Val Acc: 79.52%
Epoch [5/15] | Train Loss: 0.4358 - Train Acc: 79.87% | Val Loss: 0.4474 - Val Acc: 79.78%
Epoch [6/15] | Train Loss: 0.4102 - Train Acc: 81.00% | Val Loss: 0.4397 - Val Acc: 80.24%
Epoch [7/15] | Train Loss: 0.3879 - Train Acc: 82.67% | Val Loss: 0.4464 - Val Acc: 81.11%
Epoch [8/15] | Train Loss: 0.3710 - Train Acc: 83.51% | Val Loss: 0.4398 - Val Acc: 81.49%
Epoch [9/15] | Train Loss: 0.3509 - Train Acc: 84.55% | Val Loss: 0.4549 - Val Acc: 81.82%
Epoch [10/15] | Train Loss: 0.3358 - Train Acc: 85.46% | Val Loss: 0.4626 - Val Acc: 82.05%
Epoch [11/15] | Train Loss: 0.3218 - Train Acc: 86.03% | Val Loss: 0.4527 - Val Acc: 82.2

## Real-time Prediction Check

In [ ]:
def predict_sentiment(text,model,vocab,max_len=max_len):
  model.eval()
  cleaned=clean_text(text)
  seq=text_to_sequence(cleaned,vocab)
  padded=pad_sequence(seq,max_len)
  tensor_input=torch.tensor([padded],dtype=torch.long).to(device)

  with torch.no_grad():
    output=model(tensor_input)
    probs=torch.softmax(output,dim=1)
    confidence,predicted_class=torch.max(probs,dim=1)
  sentiment="Positive" if predicted_class.item()==1 else "Negative"
  return sentiment,confidence.item()


In [ ]:
sample_text = "The service was wonderful and amazing"
sentiment, score = predict_sentiment(sample_text, model, vocab)
print(f"Text: {sample_text}")
print(f"Prediction: {sentiment} (Confidence: {score:.2f})")

Text: The service was wonderful and amazing
Prediction: Positive (Confidence: 1.00)


In [ ]:
torch.save(model.state_dict(), 'bilstm_sentiment_model.pth')

In [ ]:
metadata = {
    'vocab': vocab,
    'max_len': max_len,
    'vocab_size': VOCAB_SIZE,
    'embedding_dim': EMBEDDING_DIM,
    'hidden_size': HIDDEN_SIZE,  # hidden_dim er bodole hidden_size rakhte paren
    'output_dim': OUTPUT_DIM,
    'pad_idx': vocab.get('<PAD>', 0) if isinstance(vocab, dict) else vocab.get_stoi().get('<PAD>', 0)
}

with open('vocab_metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

In [ ]:
def advanced_review_analysis_v3(text, model, vocab, max_len=max_len):
    sentiment, confidence = predict_sentiment(text, model, vocab, max_len)
    lower_text = text.lower()

    emotion_keywords = {
        "Joy / Excitement": ["thrilled", "amazing", "wonderful", "love", "best", "happy", "excited"],
        "Anger / Frustration": ["terrible", "awful", "worst", "hate", "angry", "horrible"],
        "Disappointment": ["disappointed", "poor", "bad", "useless"],
        "Gratitude": ["helpful", "thanks", "thank", "appreciate"]
    }

    detected_emotions = []
    found_keywords = []
    for emotion, keywords in emotion_keywords.items():
        matched = [kw for kw in keywords if kw in lower_text]
        if matched:
            detected_emotions.append(emotion)
            found_keywords.extend(matched)

    emotion_result = f"{', '.join(detected_emotions) if detected_emotions else 'Neutral'} (Keywords: {', '.join(set(found_keywords)) if found_keywords else 'None'})"

    base_impact = confidence if sentiment == "Positive" else -confidence

    aspects = {
        "Product Quality": 0.0,
        "Shipping Speed": 0.0,
        "Customer Support": 0.0,
        "Pricing / Value": 0.0
    }

    if any(w in lower_text for w in ["quality", "build", "setup", "product", "battery", "performance"]):
        aspects["Product Quality"] = round(base_impact, 2)
    if any(w in lower_text for w in ["shipping", "delivery", "arrived", "fast", "late", "speed"]):
        aspects["Shipping Speed"] = round(base_impact, 2)
    if any(w in lower_text for w in ["support", "service", "help", "helpful", "staff"]):
        aspects["Customer Support"] = round(base_impact, 2)
    if any(w in lower_text for w in ["price", "cost", "value", "expensive", "cheap", "purchase"]):
        aspects["Pricing / Value"] = round(base_impact, 2)

    active_aspects = {k: v for k, v in aspects.items() if v != 0.0}

    is_high_risk = False

    if sentiment == "Negative" and confidence > 0.90 and "Anger / Frustration" in detected_emotions:
        priority = "Critical"
        is_high_risk = True
    elif sentiment == "Negative" and (confidence > 0.80 or "Anger / Frustration" in detected_emotions):
        priority = "High"
        is_high_risk = True
    elif sentiment == "Negative" or confidence < 0.60:
        priority = "Medium"
    else:
        priority = "Low"

    if sentiment == "Positive":
        auto_replay = "Thank you so much for your glowing feedback! We're thrilled to hear that the quality and performance exceeded your expectations. We look forward to serving you again soon!"
    else:
        auto_replay = "We sincerely apologize for the issues you experienced with your recent order. Our support escalation team has been notified and we would like to issue a priority replacement or full refund right away."

    return {
        "text": text,
        "prediction": sentiment,
        "confidence": round(confidence, 4),
        "emotion": emotion_result,
        "aspect_impact": active_aspects,
        "priority_level": priority,
        "high_risk": is_high_risk,
        "auto_replay": auto_replay
    }

def generate_review_dashboard_v3(reviews_list, model, vocab):
    analyzed_results = [advanced_review_analysis_v3(r, model, vocab) for r in reviews_list]
    df_results = pd.DataFrame(analyzed_results)

    total_reviews = len(df_results)
    avg_confidence = round(df_results['confidence'].mean(), 4)
    positive_count = len(df_results[df_results['prediction'] == 'Positive'])
    positive_ratio = round((positive_count / total_reviews) * 100, 2) if total_reviews > 0 else 0.0
    high_risk_escalations = int(df_results['high_risk'].sum())

    aspect_breakdown = {
        "Product Quality": 0.0,
        "Shipping Speed": 0.0,
        "Customer Support": 0.0,
        "Pricing / Value": 0.0
    }

    for aspects in df_results['aspect_impact']:
        for asp, val in aspects.items():
            aspect_breakdown[asp] += val

    dashboard_summary = {
        "Total Analyzed Reviews": total_reviews,
        "Avg Confidence Score": avg_confidence,
        "Positive Sentiment Ratio": f"{positive_ratio}%",
        "High Risk Escalations": high_risk_escalations,
        "Aspect-Wise Sentiment Breakdown": {k: round(v, 2) for k, v in aspect_breakdown.items() if v != 0.0}
    }

    return dashboard_summary, df_results

In [ ]:
sample_text = "Absolutely thrilled with this purchase! The build quality exceeds expectations and setup was completely effortless. Battery life easily lasts 2 full days under heavy load. Customer support was extremely helpful and shipping arrived two days early!"

summary, detailed_df = generate_review_dashboard_v3([sample_text], model, vocab)

for index, row in detailed_df.iterrows():
    print(f"Review: {row['text']}\n")
    print(f"Prediction: {row['prediction']}")
    print(f"Confidence: {row['confidence']}")
    print(f"Emotion Detection: {row['emotion']}")
    print(f"Aspect Impact: {row['aspect_impact']}")
    print(f"Priority Level: {row['priority_level']}")
    print(f"Auto Replay Text: {row['auto_replay']}\n")

print("--- Dashboard Summary ---")
for key, value in summary.items():
    print(f"{key}: {value}")

Review: Absolutely thrilled with this purchase! The build quality exceeds expectations and setup was completely effortless. Battery life easily lasts 2 full days under heavy load. Customer support was extremely helpful and shipping arrived two days early!

Prediction: Positive
Confidence: 0.9885
Emotion Detection: Joy / Excitement, Gratitude (Keywords: helpful, thrilled)
Aspect Impact: {'Product Quality': 0.99, 'Shipping Speed': 0.99, 'Customer Support': 0.99, 'Pricing / Value': 0.99}
Priority Level: Low
Auto Replay Text: Thank you so much for your glowing feedback! We're thrilled to hear that the quality and performance exceeded your expectations. We look forward to serving you again soon!

--- Dashboard Summary ---
Total Analyzed Reviews: 1
Avg Confidence Score: 0.9885
Positive Sentiment Ratio: 100.0%
High Risk Escalations: 0
Aspect-Wise Sentiment Breakdown: {'Product Quality': 0.99, 'Shipping Speed': 0.99, 'Customer Support': 0.99, 'Pricing / Value': 0.99}
